# Observability and Tracing

Observability helps you understand what's happening inside your agent workflows. This tutorial covers how to add tracing and monitoring to your NAT workflows.

## What You'll Learn

1. Understanding observability in AI agents
2. Configuring Phoenix/Arize tracing
3. Viewing traces in the Phoenix UI
4. Analyzing agent behavior
5. Debugging workflow issues

## Why Observability?

- **Debug issues** - Understand why an agent made a decision
- **Monitor performance** - Track latency and token usage
- **Audit decisions** - Record agent reasoning for compliance
- **Optimize costs** - Identify expensive operations


In [1]:
import sys
from pathlib import Path

# Setup
module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


✅ Environment configured


## Prerequisites

Before using observability features, install Phoenix:

```bash
uv pip install arize-phoenix
```

Start the Phoenix server:

```bash
phoenix serve
```

This will start the Phoenix UI at `http://localhost:6006`


## Step 1: Create a Workflow


In [2]:
from nat.agent.sdk import NatReActAgent
from nat.llm.sdk import NimLLM
from nat.tool.sdk import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

# Create tools
time_tool = CurrentTimeTool(name="current_time")

try:
    from nat_simple_calculator.sdk import CalculatorToolGroup
    calculator = CalculatorToolGroup(name="calculator")
    tools = [time_tool, calculator]
except ImportError:
    tools = [time_tool]

# Create agent
agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
)

print("✅ Agent created")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✅ Agent created


## Step 2: Configure Telemetry Exporters

NAT supports Phoenix/Arize for distributed tracing. You configure this through the `NatGeneralConfiguration` with telemetry exporters and loggers:


In [3]:
from nat.observability.sdk import ConsoleLogger
from nat.observability.sdk import FileLogger
from nat.plugins.phoenix.sdk import PhoenixTelemetryExporter
from nat.utils.sdk.nat_general_configuraton import NatGeneralConfiguration

# Configure console logging
console_logger = ConsoleLogger(
    level="WARN",  # Only show warnings and errors in console
    name="console_logger",
)

# Configure file logging for detailed debugging
file_logger = FileLogger(
    path="./.tmp/workflow.log",
    level="DEBUG",  # Capture all log levels to file
    create_if_not_exists=True,
    name="file_logger",
)

# Configure Phoenix telemetry exporter for tracing
phoenix_tracer = PhoenixTelemetryExporter(
    endpoint="http://localhost:6006/v1/traces",  # Phoenix default endpoint
    project="sdk_tutorial",  # Project name in Phoenix UI
    name="phoenix_tracer",
)

# Create general configuration with observability settings
configuration = NatGeneralConfiguration(
    loggers=[console_logger, file_logger],
    telemetry_exporters=[phoenix_tracer],
)

print("✅ Telemetry configured:")
print("   📝 Console logger (WARN level)")
print("   📁 File logger (./.tmp/workflow.log)")
print("   🔭 Phoenix tracer (http://localhost:6006)")


✅ Telemetry configured:
   📝 Console logger (WARN level)
   📁 File logger (./.tmp/workflow.log)
   🔭 Phoenix tracer (http://localhost:6006)


## Step 3: Create Workflow with Configuration

Now we combine the agent with the observability configuration:


In [4]:
# Create workflow with the agent AND the observability configuration
workflow = NatWorkflow(
    entrypoint=agent,
    configuration=configuration,  # Pass the telemetry configuration
)

print("✅ Workflow created with observability enabled")


✅ Workflow created with observability enabled


## Step 4: Save Configuration

Export the workflow to YAML, including the observability settings:


In [5]:
# Save configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "traced_workflow.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Configuration saved to: {config_path}")
print("\n" + "=" * 60)
print("GENERATED CONFIGURATION WITH OBSERVABILITY:")
print("=" * 60 + "\n")

with open(config_path) as f:
    print(f.read())


📄 Configuration saved to: configs/traced_workflow.yaml

GENERATED CONFIGURATION WITH OBSERVABILITY:

general:
  telemetry:
    logging:
      console_logger:
        _type: console
        level: WARN
      file_logger:
        _type: file
        path: ./.tmp/workflow.log
        level: DEBUG
        create_if_not_exists: true
    tracing:
      phoenix_tracer:
        _type: phoenix
        project: sdk_tutorial
        endpoint: http://localhost:6006/v1/traces

functions:
  current_time:
    _type: current_datetime

function_groups:
  calculator:
    _type: calculator

llms:
  nim_llm:
    _type: nim
    model: meta/llama-3.3-70b-instruct
    max_tokens: 1024
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_time
  - calculator



## Step 5: Run with Tracing

### Via Python


In [6]:
# Run workflow - traces will be sent to Phoenix
result = await workflow.prompt("What is 25 * 4?")
print(f"Result: {result}")
print("\n💡 View traces at: http://localhost:6006")


2025-12-29 19:22:19 - ERROR    - nat.plugins.phoenix.mixin.phoenix_mixin:75 - Error exporting spans: HTTPConnectionPool(host='localhost', port=6006): Max retries exceeded with url: /v1/traces (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x35a683110>: Failed to establish a new connection: [Errno 61] Connection refused'))
Traceback (most recent call last):
  File "/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create

### Via CLI

```bash
# Make sure Phoenix is running first
phoenix serve

# Then run your workflow
nat run --config_file configs/traced_workflow.yaml \
    --input "What is 25 * 4?"

# View traces at http://localhost:6006
```


## Understanding Traces

In the Phoenix UI, you'll see:

### Trace Structure
```
📊 Workflow Execution
├── 🤖 Agent: ReActAgent
│   ├── 💭 LLM Call (reasoning)
│   │   ├── Input tokens: 150
│   │   ├── Output tokens: 50
│   │   └── Latency: 1.2s
│   ├── 🔧 Tool Call: calculator.multiply
│   │   ├── Input: {"a": 25, "b": 4}
│   │   ├── Output: 100
│   │   └── Latency: 5ms
│   └── 💭 LLM Call (final response)
│       ├── Input tokens: 200
│       ├── Output tokens: 30
│       └── Latency: 0.8s
└── ✅ Result: "25 * 4 = 100"
```

### Key Metrics
- **Latency** - Time taken for each operation
- **Token Usage** - Input/output tokens per LLM call
- **Tool Calls** - Which tools were invoked and their results
- **Error Traces** - Failed operations and error messages


## Debugging with Traces

Traces help you debug common issues:

### Issue: Agent Not Using Tools
Look at the LLM reasoning spans to see if:
- Tools are mentioned in the prompt
- Tool descriptions are clear
- Agent understood the task

### Issue: Slow Performance
Check latency breakdown:
- LLM calls are usually the slowest
- Tool calls should be fast
- Network issues show high latency

### Issue: Wrong Answers
Examine the reasoning chain:
- What tools were called?
- What were the tool results?
- How did the LLM interpret results?


## Summary

In this tutorial, you learned:

✅ Why observability matters for AI agents  
✅ Configuring loggers (`ConsoleLogger`, `FileLogger`)  
✅ Setting up Phoenix telemetry exporter (`PhoenixTelemetryExporter`)  
✅ Combining observability with `NatGeneralConfiguration`  
✅ Exporting traced workflows to YAML  
✅ Understanding trace structure and debugging  

## Next Steps

- **[10_evaluation.ipynb](./10_evaluation.ipynb)** - Evaluate workflow performance
- **[09_configuration_guide.ipynb](./09_configuration_guide.ipynb)** - Deep dive into configuration

## Additional Resources

- [Phoenix Documentation](https://docs.arize.com/phoenix)
- [OpenTelemetry Tracing](https://opentelemetry.io/docs/)
